In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

#Load Data

In [0]:
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
df = spark.read.parquet(f"dbfs:/mnt/mids-w261/OTPW_60M_Backup/")

In [0]:
display(df)

#Simple Data Metrics

In [0]:
# rows and cols
print(f'# of Columns: {len(df.columns)}')
print(f'# of Rows: {df.count()}')


In [0]:
# size of data
size_bytes = sum(file.size for file in dbutils.fs.ls("dbfs:/mnt/mids-w261/OTPW_60M_Backup/"))
size_gb = size_bytes / (1024**3)

print(f"{size_gb} GB")

In [0]:
# schema
df.printSchema()

In [0]:
# of cancelled flights, we will remove the cancelled flights from our analysis
df.select((F.avg(F.col("CANCELLED").cast("double")) * 100).alias("cancelled_pct")).display()

In [0]:
# missing values
total = df.count()

missing_row = df.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
    for c in df.columns
])

missing_long = missing_row.selectExpr(
    "stack({0}, {1}) as (column, missing_pct)".format(
        len(df.columns),
        ", ".join([f"'{c}', `{c}`" for c in df.columns])
    )
)

display(missing_long.orderBy(F.desc("missing_pct")))

In [0]:
high_missing = missing_long.filter(F.col("missing_pct") > 50)

display(high_missing.orderBy(F.desc("missing_pct")))

In [0]:
high_missing_count = high_missing.count()

print(f'Count: {high_missing_count}')
print(f'Percent: {high_missing_count/len(df.columns)}')

In [0]:
# filter out cancelled flights
df = df.filter(F.col("CANCELLED").cast("double") == 0)

before = df.count()
# filter for only FM-15 report type
df = df.filter(F.col("REPORT_TYPE") == "FM-15")

# filter missing departure delay
df= df.filter(
    F.col("DEP_DEL15").isNotNull() & (~F.isnan(F.col("DEP_DEL15")))
)

after=df.count()

print(after/before)

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# --- existing aggregation ---
plot_df = (
    df.select(F.col("DEP_DEL15").cast("double").alias("DEP_DEL15"))
      .filter(F.col("DEP_DEL15").isNotNull())
      .groupBy("DEP_DEL15")
      .count()
      .orderBy("DEP_DEL15")
      .toPandas()
)

# map labels
plot_df["label"] = plot_df["DEP_DEL15"].map({
    0.0: "On-time",
    1.0: "Delayed"
})

# --- compute percentages ---
total = plot_df["count"].sum()
plot_df["pct"] = plot_df["count"] / total * 100

# --- plot ---
plt.figure(figsize=(6,4))
bars = plt.bar(plot_df["label"], plot_df["count"])

# --- annotate percentages on bars ---
for bar, pct in zip(bars, plot_df["pct"]):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{pct:.1f}%",
        ha='center',
        va='bottom'
    )

# --- axis labels ---
plt.xlabel("Flight Status")
plt.ylabel("Number of Flights")
plt.title("Flight Delay Distribution (DEP_DEL15)")

# --- force full numbers on y-axis (no scientific notation) ---
plt.gca().yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

plt.tight_layout()
plt.show()

In [0]:
# -------------------------
# Feature engineering
# -------------------------
df2 = df.withColumn(
    "hour",
    (F.col("CRS_DEP_TIME").cast("int") / 100).cast("int")
).withColumn(
    "DEP_DEL15",
    F.col("DEP_DEL15").cast("double")
).withColumn(
    "DAY_OF_WEEK",
    F.col("DAY_OF_WEEK").cast("int")
).withColumn(
    "MONTH",
    F.col("MONTH").cast("int")
)

# -------------------------
# Aggregations
# -------------------------
hour_df = df2.groupBy("hour").agg(F.avg("DEP_DEL15").alias("delay_rate")).orderBy("hour")
dow_df = df2.groupBy("DAY_OF_WEEK").agg(F.avg("DEP_DEL15").alias("delay_rate")).orderBy("DAY_OF_WEEK")
month_df = df2.groupBy("MONTH").agg(F.avg("DEP_DEL15").alias("delay_rate")).orderBy("MONTH")

# -------------------------
# Pandas
# -------------------------
hour_pd = hour_df.toPandas()
dow_pd = dow_df.toPandas()
month_pd = month_df.toPandas()

# -------------------------
# Plot
# -------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Hour
axes[0].plot(hour_pd["hour"], hour_pd["delay_rate"], marker="o")
axes[0].set_title("Delay Rate by Hour of Day")
axes[0].set_xlabel("Hour")
axes[0].set_xticks(hour_pd["hour"])

# Day of Week
axes[1].plot(dow_pd["DAY_OF_WEEK"], dow_pd["delay_rate"], marker="o")
axes[1].set_title("Delay Rate by Day of Week")
axes[1].set_xlabel("Day of Week")
axes[1].set_xticks(dow_pd["DAY_OF_WEEK"])

# Month
axes[2].plot(month_pd["MONTH"], month_pd["delay_rate"], marker="o")
axes[2].set_title("Delay Rate by Month")
axes[2].set_xlabel("Month")
axes[2].set_xticks(month_pd["MONTH"])

plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import when, col
df = df.withColumn(
    "carrier_name",
    F.when(F.col("OP_UNIQUE_CARRIER") == "B6", "JetBlue")
    .when(F.col("OP_UNIQUE_CARRIER") == "F9", "Frontier")
    .when(F.col("OP_UNIQUE_CARRIER") == "VX", "Virgin America")
    .when(F.col("OP_UNIQUE_CARRIER") == "WN", "Southwest")
    .when(F.col("OP_UNIQUE_CARRIER") == "NK", "Spirit")
    .when(F.col("OP_UNIQUE_CARRIER") == "G4", "Allegiant")
    .when(F.col("OP_UNIQUE_CARRIER") == "OH", "PSA Airlines")
    .when(F.col("OP_UNIQUE_CARRIER") == "UA", "United")
    .when(F.col("OP_UNIQUE_CARRIER") == "YV", "Mesa Airlines")
    .when(F.col("OP_UNIQUE_CARRIER") == "MQ", "Envoy Air")
    .otherwise(F.col("OP_UNIQUE_CARRIER"))
)

In [0]:
# ---------------- Clean delay column ----------------
df2 = df.withColumn("DEP_DEL15", F.col("DEP_DEL15").cast("double"))

# ---------------- Carrier (TOP 10 by volume-aware delay rate) ----------------
carrier_df = (
    df2.groupBy("carrier_name")
       .agg(
           F.avg("DEP_DEL15").alias("delay_rate"),
           F.count("*").alias("num_flights")
       )
       .orderBy("delay_rate", ascending=False)
)

carrier_pd = carrier_df.toPandas().head(10)

# ---------------- Origin (filter low-volume airports) ----------------
origin_df = (
    df2.groupBy("ORIGIN")
       .agg(
           F.avg("DEP_DEL15").alias("delay_rate"),
           F.count("*").alias("num_flights")
       )
       .filter(F.col("num_flights") > 1000)   # IMPORTANT FIX
       .orderBy("delay_rate", ascending=False)
)

origin_pd = origin_df.toPandas().head(10)

# ---------------- Plot ----------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Carrier
axes[0].bar(carrier_pd["carrier_name"], carrier_pd["delay_rate"])
axes[0].set_title("Top 10 Airline Delay Rates")
axes[0].set_xlabel("Airline")
axes[0].set_ylabel("Delay Rate")
axes[0].tick_params(axis='x', rotation=45)

# Origin
axes[1].bar(origin_pd["ORIGIN"], origin_pd["delay_rate"])
axes[1].set_title("Top 10 Origin Airports (>1000 flights)")
axes[1].set_xlabel("Airport")
axes[1].set_ylabel("Delay Rate")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

#Correlation

In [0]:
from pyspark.sql.functions import col

In [0]:
columns = [
    "DEP_DEL15",
    "DISTANCE",
    "CRS_ELAPSED_TIME",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature",
    "HourlyWindSpeed",
    "HourlyWindGustSpeed",
    "HourlyWindDirection",
    "HourlyVisibility",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlyStationPressure",
    "HourlySeaLevelPressure"
]

df_casted = df.select([col(c).cast("double").alias(c) for c in columns])

In [0]:
from pyspark.sql.functions import expr
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.functions import col
import matplotlib.pyplot as plt

weather_cols = [
    "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature",
    "HourlyWindSpeed",
    "HourlyWindGustSpeed",
    "HourlyVisibility",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlyStationPressure",
    "HourlySeaLevelPressure"
]

df_clean = df2.select(
    "*",
    *[expr(f"try_cast({c} as double)").alias(c + "_d") for c in weather_cols]
)

conditions = [
    col(c + "_d").isNotNull()
    for c in weather_cols
]

df_clean = df_clean.filter(reduce(lambda a, b: a & b, conditions))

features = [c + "_d" for c in weather_cols]

means_df = df_clean.groupBy("DEP_DEL15").agg(
    *[F.avg(c).alias(c) for c in features]
)

pdf = means_df.toPandas().set_index("DEP_DEL15")

ontime = pdf.loc[0]
delayed = pdf.loc[1]

# ---------------- % difference ----------------
pct_diff = ((delayed - ontime) / ontime) * 100

# 🔥 OPTIONAL: sort by absolute impact (largest effect first)
pct_diff = pct_diff.reindex(pct_diff.abs().sort_values(ascending=False).index)

plt.figure(figsize=(12,6))

bars = plt.bar(pct_diff.index, pct_diff.values)

plt.axhline(0, color="black", linewidth=1)
plt.xticks(rotation=45, ha="right")
plt.ylabel("% Difference (Delayed vs On-Time)")
plt.title("Weather Differences Between Delayed and On-Time Flights")

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{height:.1f}%",
        ha="center",
        va="bottom" if height > 0 else "top"
    )

plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import expr
from functools import reduce
from pyspark.sql.functions import col

weather_cols = [
    "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature",
    "HourlyWindSpeed",
    "HourlyWindGustSpeed",
    "HourlyVisibility",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlyStationPressure",
    "HourlySeaLevelPressure"
]

df_clean = df2.select(
    "*",
    *[expr(f"try_cast({c} as double)").alias(c + "_d") for c in weather_cols]
)

conditions = [
    col(c + "_d").isNotNull()
    for c in weather_cols
]

df_clean = df_clean.filter(reduce(lambda a, b: a & b, conditions))

means_df = df_clean.groupBy("DEP_DEL15").agg(
    *[F.avg(c).alias(c) for c in features]
)

pdf = means_df.toPandas().set_index("DEP_DEL15")

ontime = pdf.loc[0]
delayed = pdf.loc[1]

pct_diff = ((delayed - ontime) / ontime) * 100


plt.figure(figsize=(12,6))

bars = plt.bar(features, pct_diff)

plt.axhline(0, color="black", linewidth=1)
plt.xticks(rotation=45, ha="right")
plt.ylabel("% Difference (Delayed vs On-Time)")
plt.title("Weather Differences Between Delayed and On-Time Flights")

# add labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{height:.1f}%",
        ha="center",
        va="bottom" if height > 0 else "top"
    )

plt.tight_layout()
plt.show()

In [0]:
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041726_otpw_train_df.parquet")
otpw_test_df = spark.read.parquet(f"{TEAM_DIR}/041726_otpw_test_df.parquet")

df = otpw_train_df.unionByName(otpw_test_df)

In [0]:
from pyspark.sql import functions as F

# counts
train_count = otpw_train_df.count()
test_count  = otpw_test_df.count()
total_count = train_count + test_count

# create summary DataFrame
summary_df = spark.createDataFrame(
    [("OTPW Flights", total_count, train_count, test_count)],
    ["Dataset", "Total Rows", "Train Rows", "Test Rows"]
)

summary_df.show(truncate=False)

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pyspark.sql import functions as F

# Convert to Pandas (small aggregation only)
plot_df = (
    df.groupBy("IsWeekend")
      .count()
      .orderBy("IsWeekend")
      .toPandas()
)

# Map labels
plot_df["label"] = plot_df["IsWeekend"].map({
    0: "Weekday",
    1: "Weekend"
})

# Plot
plt.figure(figsize=(6,4))
bars = plt.bar(plot_df["label"], plot_df["count"])

# Add percentages
total = plot_df["count"].sum()
for bar, count in zip(bars, plot_df["count"]):
    pct = count / total * 100
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{pct:.1f}%",
        ha='center',
        va='bottom'
    )

# Labels
plt.xlabel("Day Type")
plt.ylabel("Number of Flights")
plt.title("Flights by Weekend vs Weekday")

# Full numbers on y-axis
plt.gca().yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql import functions as F

summary_df = (
    df.groupBy("DEP_DEL15")
      .agg(
          # ---- percentage feature ----
          F.count("delay_pct_flights_2_4hr_before").alias("n_rows"),
          F.mean("delay_pct_flights_2_4hr_before").alias("mean_delay_pct"),
          F.stddev("delay_pct_flights_2_4hr_before").alias("std_delay_pct"),
          F.min("delay_pct_flights_2_4hr_before").alias("min_delay_pct"),
          F.expr("percentile_approx(delay_pct_flights_2_4hr_before, 0.25)").alias("p25_delay_pct"),
          F.expr("percentile_approx(delay_pct_flights_2_4hr_before, 0.5)").alias("median_delay_pct"),
          F.expr("percentile_approx(delay_pct_flights_2_4hr_before, 0.75)").alias("p75_delay_pct"),
          F.max("delay_pct_flights_2_4hr_before").alias("max_delay_pct"),

          # ---- count feature ----
          F.mean("delayed_flights_2_4hr_before").alias("mean_delay_count"),
          F.stddev("delayed_flights_2_4hr_before").alias("std_delay_count"),
          F.min("delayed_flights_2_4hr_before").alias("min_delay_count"),
          F.expr("percentile_approx(delayed_flights_2_4hr_before, 0.25)").alias("p25_delay_count"),
          F.expr("percentile_approx(delayed_flights_2_4hr_before, 0.5)").alias("median_delay_count"),
          F.expr("percentile_approx(delayed_flights_2_4hr_before, 0.75)").alias("p75_delay_count"),
          F.max("delayed_flights_2_4hr_before").alias("max_delay_count")
      )
      .orderBy("DEP_DEL15")
)

summary_df.display()

In [0]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
import matplotlib.ticker as mticker

# ---------------- Clean data ----------------
clean_df = df.select("DEP_DEL15", "prev_DEP_DEL15") \
             .dropna()

# ---------------- Aggregate ----------------
plot_df = (
    clean_df.groupBy("prev_DEP_DEL15")
            .agg(
                F.mean("DEP_DEL15").alias("delay_rate"),
                F.count("*").alias("count")
            )
            .orderBy("prev_DEP_DEL15")
            .toPandas()
)

# ---------------- Labels ----------------
plot_df["label"] = plot_df["prev_DEP_DEL15"].map({
    0: "Previous On-time",
    1: "Previous Delayed"
})

plot_df["delay_pct"] = plot_df["delay_rate"] * 100

# ---------------- Plot ----------------
plt.figure(figsize=(6,4))
bars = plt.bar(plot_df["label"], plot_df["delay_rate"])

# ---------------- Percent annotations ----------------
for bar, pct in zip(bars, plot_df["delay_pct"]):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{pct:.1f}%",
        ha="center",
        va="bottom"
    )

# ---------------- Styling ----------------
plt.title("Current Flight Delay Rate vs Previous Flight Status")
plt.xlabel("Previous Flight Status")
plt.ylabel("Delay Rate")

plt.gca().yaxis.set_major_formatter(
    mticker.StrMethodFormatter("{x:.2f}")
)

plt.ylim(0, plot_df["delay_rate"].max() * 1.2)
plt.tight_layout()
plt.show()

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

# ---------------- Bounds ----------------
MAX_TURNAROUND = 300  # adjust as needed

# ---------------- Filter ----------------
sample = (
    df.select("turnaround_minutes", "DEP_DEL15")
      .dropna()
      .filter(
          (F.col("turnaround_minutes") >= 0) &
          (F.col("turnaround_minutes") <= MAX_TURNAROUND)
      )
      .sample(0.1, seed=42)
      .toPandas()
)

# ---------------- Plot ----------------
plt.figure(figsize=(7,4))

plt.hist(
    [
        sample[sample.DEP_DEL15 == 0]["turnaround_minutes"],
        sample[sample.DEP_DEL15 == 1]["turnaround_minutes"]
    ],
    bins=40,
    label=["On-time", "Delayed"],
    alpha=0.6
)

plt.legend()
plt.xlabel("Turnaround Minutes (0–300)")
plt.ylabel("Count")
plt.title("Turnaround Time Distribution by Delay Status")

plt.show()

In [0]:
df.display()

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- Features ----------------
features = [
    "DryBulbTemperature_trend_2h_6h",
    "WindSpeed_trend_2h_6h",
    "RelativeHumidity_trend_2h_6h",
    "Visibility_trend_2h_6h",
    "2h_HourlyDryBulbTemperature",
    "2h_HourlyWindSpeed",
    "2h_HourlyRelativeHumidity",
    "VisibilityCat",
    "PrecipBinary",
    "DEP_DEL15"
]

# ---------------- Data prep ----------------
pdf = (
    df.select(features)
      .dropna()
      .toPandas()
)

# ---------------- Spearman correlation ----------------
corr_matrix = pdf.corr(method="spearman")

# ---------------- Heatmap ----------------
plt.figure(figsize=(10,8))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5
)

plt.title("Spearman Correlation Heatmap (Clipped to [-1, 1])")
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

# ---------------- Set limit ----------------
MAX_WIND_SPEED = 60  # mph (adjust if needed)

# ---------------- Sample + filter ----------------
sample = (
    df.select(
        "2h_HourlyDryBulbTemperature",
        "2h_HourlyWindSpeed",
        "2h_HourlyRelativeHumidity",
        "VisibilityCat",
        "PrecipBinary"
    )
    .dropna()
    .filter(F.col("2h_HourlyWindSpeed") <= MAX_WIND_SPEED)
    .sample(0.1, seed=42)
    .toPandas()
)

# ---------------- Figure ----------------
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

# 1. Temperature
axes[0].hist(sample["2h_HourlyDryBulbTemperature"], bins=30)
axes[0].set_title("Temperature (°F)")

# 2. Wind Speed (filtered)
axes[1].hist(sample["2h_HourlyWindSpeed"], bins=30)
axes[1].set_title("Wind Speed (mph, ≤ 60)")

# 3. Humidity
axes[2].hist(sample["2h_HourlyRelativeHumidity"], bins=30)
axes[2].set_title("Relative Humidity (%)")

# 4. Visibility Category
sns.countplot(x="VisibilityCat", data=sample, ax=axes[3])
axes[3].set_title("Visibility Category")

# 5. Precipitation
sns.countplot(x="PrecipBinary", data=sample, ax=axes[4])
axes[4].set_title("Precipitation (Binary)")

# 6. Empty subplot
axes[5].axis("off")

# ---------------- Layout ----------------
plt.suptitle("Weather Feature Distributions (Wind Speed Filtered)", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

# ---------------- Limits ----------------
MAX_TEMP_TREND = 20
MAX_WIND_TREND = 30
MAX_HUMIDITY_TREND = 50
MAX_VIS_TREND = 10

# ---------------- Sample + filter ----------------
sample = (
    df.select(
        "DryBulbTemperature_trend_2h_6h",
        "WindSpeed_trend_2h_6h",
        "RelativeHumidity_trend_2h_6h",
        "Visibility_trend_2h_6h"
    )
    .dropna()
    .filter(
        (F.col("DryBulbTemperature_trend_2h_6h").between(-MAX_TEMP_TREND, MAX_TEMP_TREND)) &
        (F.col("WindSpeed_trend_2h_6h").between(-MAX_WIND_TREND, MAX_WIND_TREND)) &
        (F.col("RelativeHumidity_trend_2h_6h").between(-MAX_HUMIDITY_TREND, MAX_HUMIDITY_TREND)) &
        (F.col("Visibility_trend_2h_6h").between(-MAX_VIS_TREND, MAX_VIS_TREND))
    )
    .sample(0.1, seed=42)
    .toPandas()
)

# ---------------- Plot ----------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

# 1. Temperature trend
axes[0].hist(sample["DryBulbTemperature_trend_2h_6h"], bins=30)
axes[0].set_title("Temperature Trend (°F, ±20 cap)")

# 2. Wind speed trend
axes[1].hist(sample["WindSpeed_trend_2h_6h"], bins=30)
axes[1].set_title("Wind Speed Trend (mph, ±30 cap)")

# 3. Humidity trend
axes[2].hist(sample["RelativeHumidity_trend_2h_6h"], bins=30)
axes[2].set_title("Humidity Trend (% change, ±50 cap)")

# 4. Visibility trend
axes[3].hist(sample["Visibility_trend_2h_6h"], bins=30)
axes[3].set_title("Visibility Trend (mi, ±10 cap)")

# ---------------- Layout ----------------
plt.suptitle("Weather Trend Features (Capped Distributions)", fontsize=14)
plt.tight_layout()
plt.show()